## Multi-agents using Router pattern 

In [30]:
"""
Router pattern with Azure OpenAI — plain LangChain (no LangGraph code)
======================================================================

    query ─> router ─┬─> orders agent ──┐
                     ├─> billing agent ─┼─> synthesize ─> answer
                     └─> tech agent ────┘
            (only the relevant agents run, in parallel)

Run:
    pip install -r requirements.txt
    cp .env.example .env   # fill in your Azure values
    python router_simple.py
"""


'\nRouter pattern with Azure OpenAI — plain LangChain (no LangGraph code)\n======================================================================\n\n    query ─> router ─┬─> orders agent ──┐\n                     ├─> billing agent ─┼─> synthesize ─> answer\n                     └─> tech agent ────┘\n            (only the relevant agents run, in parallel)\n\nRun:\n    pip install -r requirements.txt\n    cp .env.example .env   # fill in your Azure values\n    python router_simple.py\n'

In [31]:
import os
from typing import Literal

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_openai import AzureChatOpenAI
from pydantic import BaseModel, Field


In [32]:
load_dotenv()


True

In [33]:
# ---------------------------------------------------------------------------
# 1. Azure OpenAI models
#    Reads AZURE_OPENAI_API_KEY, AZURE_OPENAI_ENDPOINT, OPENAI_API_VERSION from env.
# ---------------------------------------------------------------------------

agent_llm = ChatOpenAI(
    model=os.getenv("CHAT_MODEL_DEPLOYMENT_NAME"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)
router_llm = ChatOpenAI(
    model=os.getenv("CHAT_MODEL_DEPLOYMENT_NAME"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)


In [34]:
# ---------------------------------------------------------------------------
# 2. Tools for each specialist
# ---------------------------------------------------------------------------
@tool
def get_order_status(order_id: str) -> str:
    """Get status, carrier, tracking number and ETA for an order."""
    orders = {
        "A1001": "Shipped via BlueDart, tracking BD778812, ETA Oct 30",
        "A1002": "Processing, ETA Nov 2",
    }
    return orders.get(order_id.upper(), f"No order found with id {order_id}")


@tool
def get_return_policy() -> str:
    """Get the return policy."""
    return "30-day returns for unused items; electronics 15 days with all accessories."


@tool
def get_invoice(order_id: str) -> str:
    """Get the charges for an order."""
    invoices = {
        "A1001": "Headphones ₹4,499, charged twice on Oct 24 (second charge is a duplicate authorization)",
        "A1002": "Smart watch ₹8,999, charged once on Oct 26",
    }
    return invoices.get(order_id.upper(), f"No invoice for {order_id}")


@tool
def get_refund_policy() -> str:
    """Get refund timelines and how duplicate charges are handled."""
    return "Duplicate authorizations auto-reverse in 5–7 business days. Card refunds 5–7 days, UPI 1–3 days."


@tool
def search_troubleshooting(issue: str) -> str:
    """Search troubleshooting articles for a product issue."""
    return (
        "1) Forget the device on your phone and re-pair (hold power 7s). "
        "2) Update to firmware v2.3.1, which fixes Android 15 dropouts. "
        "3) Factory reset: hold power + vol- for 10s."
    )


In [35]:
# ---------------------------------------------------------------------------
# 3. Specialized agents
# ---------------------------------------------------------------------------
orders_agent = create_agent(
    agent_llm,
    tools=[get_order_status, get_return_policy],
    system_prompt="You are the Orders & Shipping specialist. Answer using your tools only.",
)
billing_agent = create_agent(
    agent_llm,
    tools=[get_invoice, get_refund_policy],
    system_prompt="You are the Billing specialist. Explain charges and refunds using your tools only.",
)
tech_agent = create_agent(
    agent_llm,
    tools=[search_troubleshooting],
    system_prompt="You are the Technical Support specialist. Give clear numbered steps.",
)

agents = {
    "orders": orders_agent,
    "billing": billing_agent,
    "tech": tech_agent,
}

In [36]:
# ---------------------------------------------------------------------------
# 4. Router: decide which agents to call, each with its own sub-question
# ---------------------------------------------------------------------------
class Route(BaseModel):
    agent: Literal["orders", "billing", "tech"]
    question: str = Field(description="Self-contained sub-question for this agent")


class RouterDecision(BaseModel):
    routes: list[Route] = Field(description="Relevant agents only; empty if none apply")


ROUTER_PROMPT = """Route this customer-support message to specialist agents.
- orders:  order status, shipping, tracking, returns
- billing: charges, invoices, duplicate charges, refunds
- tech:    product problems, pairing, firmware, troubleshooting
For each relevant agent, write a self-contained sub-question (include order ids, product names).
Return an empty list for small talk."""

router = router_llm.with_structured_output(RouterDecision)


In [37]:
def ask_agent(name: str, question: str) -> str:
    result = agents[name].invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content

In [38]:
# ---------------------------------------------------------------------------
# 5. The whole flow: route -> run agents in parallel -> synthesize
# ---------------------------------------------------------------------------
def answer(query: str) -> str:
    # Step 1: route
    decision = router.invoke([("system", ROUTER_PROMPT), ("user", query)])
    if not decision.routes:
        return "I can help with orders, billing, or product issues. What do you need?"

    print("Router picked:")
    for r in decision.routes:
        print(f"  - {r.agent}: {r.question}")

    # Step 2: run the chosen agents in parallel
    parallel = RunnableParallel(
        {
            r.agent: RunnableLambda(lambda _, r=r: ask_agent(r.agent, r.question))
            for r in decision.routes
        }
    )
    findings = parallel.invoke({})

    # Step 3: combine into one reply
    notes = "\n\n".join(f"[{name.upper()}]\n{text}" for name, text in findings.items())
    reply = router_llm.invoke(
        [
            ("system", "You are a friendly support lead. Write one reply that answers every part "
                       "of the customer's message using only the specialist notes."),
            ("user", f"Customer message: {query}\n\nSpecialist notes:\n{notes}"),
        ]
    )
    return reply.content


In [ ]:

questions = [
    "My headphones from order A1001 keep disconnecting from my Android phone, "
    "I was charged twice, and what's the return policy?",
    "Where is order A1002?",
    "hi there!",
]
for q in questions:
    print("=" * 70)
    print("CUSTOMER:", q, "\n")
    print("\nREPLY:\n" + answer(q))

CUSTOMER: My headphones from order A1001 keep disconnecting from my Android phone, I was charged twice, and what's the return policy? 

Router picked:
  - tech: Troubleshoot the headphones from order A1001, which keep disconnecting from the customer's Android phone.
  - billing: Investigate the duplicate charge associated with order A1001 and explain the refund process for the extra charge.
  - orders: Explain the return policy and return options for the headphones from order A1001.
